# 第 11 章 CellTypist 自动注释

## 学习目标

学习自动注释，并用标记基因和手工结果检查模型预测。

## 为什么做与怎样做

确认实际适配模型后，从 counts 构造 10,000 counts log1p 输入；并列保留手工/模型标签，复核分歧后确认报告主标签。

前置章节：10。运行前请完成项目环境准备。

本章在项目副本运行。遇到等待材料/确认，请按根目录 **99_运行与AI协作指南.md** 查看当前结果、与用户讨论并续跑；不能跳过待办直接进入下游。


In [ ]:
# 功能说明：查找公共环境和当前项目，读取有效上游检查点；模板副本之间不共享分析状态。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
from __future__ import annotations
from pathlib import Path
import os
import sys
ROOT = Path(os.environ.get("SC_COURSE_ROOT", Path.cwd())).resolve()
while not (ROOT / "config/course.json").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "config/course.json").is_file():
    raise RuntimeError("请从课程目录或其 notebooks/exercises 目录运行。")
os.environ["CELLTYPIST_FOLDER"] = str(ROOT / ".runtime/celltypist")
os.environ["MPLCONFIGDIR"] = str(ROOT / ".runtime/matplotlib")
sys.path.insert(0, str(ROOT / "tools"))
from course_runtime import start_chapter, marker_sets, scaled_view
from course_projects import resolve_project, input_path, read_sample
from course_resources import qc_gene_sets, marker_resources, resolution_preview
import anndata as ad
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
from scipy.sparse import csr_matrix
from scipy.stats import median_abs_deviation
INSTALL_ROOT = ROOT
ROOT = resolve_project()
ctx = start_chapter("11")
adata = ctx.load_input()
coarse_key = str(adata.uns["annotation_keys"]["coarse"])
fine_key = str(adata.uns["annotation_keys"]["fine"])




## 11.1 自动注释

In [ ]:
# 功能说明：更新图形输出目录。
# 运行目的：为自动注释部分设置单独的图片保存路径。

# 设置输出目录
sc.settings.figdir = ctx.figures

与手动注释数据方法不同，如下所述，生成的注释质量可能参差不齐。因此，重要的是将这些方法视为注释过程的起点而不是终点。
如前所述，自动生成的注释质量可能会有所不同。更具体地说，注释的质量取决于：
1) 分类器训练数据的质量。如果训练数据没有得到很好的注释或注释分辨率很低，分类器也会这样做。同样，如果训练数据和/或其注释有噪声，分类器可能表现不佳。<br>
2) 您自己的数据与分类器训练数据的相似性。例如，如果分类器是在 drop-seq 单细胞数据集上训练的，而您的数据是 10X 单核而不是单细胞 drop-seq，这可能会降低注释的质量。在包含多种数据集的跨数据集图谱上训练的分类器可能比在单个数据集上训练的分类器提供更稳健和质量更好的注释。一个例子是在人类肺细胞图谱 （参考文献：anno:Sikkema2023） 上训练的 CellTypist（一种将在下面更广泛讨论的自动注释方法）分类器，其中包括 14 个不同的肺数据集。该模型在新肺数据上的表现可能优于在单个肺数据集上训练的模型。

上述几点强调了使用分类器的可能缺点，具体取决于训练数据和模型类型。尽管如此，使用预训练分类器注释数据有几个重要的优点。首先，这是一种快速简便的注释数据的方法。注释不需要下载也不需要预处理训练数据，有时只涉及将数据上传到在线网页。其次，这些方法不依赖于将数据划分为簇，就像手动注释那样。第三，预训练分类器使您能够直接利用先前研究的知识和信息，例如高质量的注释。最后，使用此类分类器有助于协调整个领域的细胞类型定义，从而为就这些定义达成全领域共识扫清道路。

值得注意的是，到目前为止讨论的方法仅使用了数据中检测到的一小部分基因：通常每个细胞类型仅使用 1 到 ~10 个标记基因。另一种方法是使用将更大的基因集（数千个或更多）作为输入的分类器，从而更多地利用 scRNA-seq 数据的广度。此类分类器在先前注释的数据集或图谱上进行训练。这些示例包括 CellTypist （参考文献：anno:Conde2022）（另请参见 https://www.celltypist.org，可以将数据上传到门户以获得自动细胞注释）和 Clustifyr （参考文献：anno:Fu2020）。

让我们在我们的数据上试用 CellTypist。根据 CellTypist 教程 (https://www.celltypist.org/tutorials;https://www.celltypist.org/models)，我们知道我们需要准备数据，以便将计数归一化为每个细胞 10,000 个计数，然后进行 log1p 转换：

In [ ]:
# 从 counts 构造模型要求的输入；主分析对象的 log1p 层保持原含义。
adata_celltypist = adata.copy()
adata_celltypist.X = adata_celltypist.layers["counts"].copy()
sc.pp.normalize_total(adata_celltypist, target_sum=10**4)
sc.pp.log1p(adata_celltypist)
# CellTypist 接受稀疏 AnnData；模型内部仅对所需特征做缩放。

本课程使用随包提供的免疫细胞 CellTypist 模型，可直接从 model 目录加载。

In [ ]:
# 功能说明：导入 CellTypist 库。
# 运行目的：加载自动注释所需的工具包。
# 详细代码解析：
# - `import celltypist`: 导入主包。
# - `from celltypist import models`: 导入模型管理模块。

# import sys
# !{sys.executable} -m pip install celltypist

import celltypist
from celltypist import models

In [ ]:
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
print(ROOT / "inputs/models")



In [ ]:
# 变量/函数/参数解析：
# - model_specs/model_paths：本项目登记的模型元数据和解析后的本地路径。
# - ctx.choose("models", ...)：先说明物种、组织、标签范围及来源，等待用户确认。
# - 下载/跳过：由 99 指南协调；没有模型不会自动套用人类免疫答案。
# 功能说明：实际模型先经范围核对和用户确认，不能默认沿用教程免疫模型。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
model_specs = ctx.config["models"]
model_paths = [input_path(ROOT, spec["path"]) for spec in model_specs]
ctx.choose("models", {"use": {"models": model_specs}}, "请核对已找到模型的物种、组织、标签范围和来源，再确认 use；没有适配模型可注明原因跳过本章。")
for path in model_paths:
    print(path.name, path.stat().st_size, "bytes")


让我们尝试 `Immune_All_Low` 和 `Immune_All_High` 模型（这些分别在更精细的注释级别（低）和更粗略的级别（高）注释免疫细胞类型）：

In [ ]:
# 变量/函数/参数解析：
# - coarse/fine：模型角色；仅存在一个模型时只生成相应结果，不伪造另一层。
# - overlap：模型特征与本次基因名称的交集数，零交集会暂停要求核对编号。
# - 交集非零不代表模型适配，仍需结合物种、组织、标签和后续预测检查。
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
loaded_models = {spec["name"]: models.Model.load(str(path)) for spec,path in zip(model_specs,model_paths)}
model_high = loaded_models.get("coarse")
model_low = loaded_models.get("fine")
if model_high is None and model_low is None:
    model_low = next(iter(loaded_models.values()))
for model in loaded_models.values():
    overlap = len(set(model.features) & set(adata.var_names))
    print("模型特征匹配：", overlap, "/", len(model.features))
    if overlap == 0:
        ctx.wait_input("model_genes", "模型与数据没有共同基因，请核对编号映射和适用范围。")



对于其中的每一个，我们可以查看它包含哪些细胞类型，看看是否包含骨髓细胞类型：

In [ ]:
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
if model_high is not None:
    # 功能说明：查看粗略模型包含的细胞类型。
    # 运行目的：了解该模型能识别哪些细胞类型，确认是否包含我们感兴趣的类型。
    # 详细代码解析：
    # 1. `model_high.cell_types`
    #    - 打印模型中定义的所有细胞类型标签列表。

    model_high.cell_types


In [ ]:
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
if model_low is not None:
    # 功能说明：查看精细模型包含的细胞类型。
    # 运行目的：了解精细模型能识别哪些具体的细胞亚型。

    model_low.cell_types


看起来这些模型包括许多不同的免疫细胞类型祖细胞！
现在让我们运行模型。首先是粗略的那个：

In [ ]:
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
if model_high is not None:
    # 功能说明：使用粗略模型进行细胞类型预测。
    # 运行目的：自动注释数据中的细胞类型（粗略级别）。
    # 详细代码解析：
    # 1. `celltypist.annotate(...)`
    #    - `adata_celltypist`: 预处理好的数据对象。
    #    - `model=model_high`: 使用粗略模型。
    #    - `majority_voting=True`:
    #      - 启用多数投票机制。
    #      - 结合细胞周围邻居的预测结果来修正当前细胞的预测，通常能提高准确性并减少噪声。

    predictions_high = celltypist.annotate(
        adata_celltypist, model=model_high, majority_voting=True
    )


将预测转换为 adata 以获得完整输出...

In [ ]:
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
if model_high is not None:
    # 功能说明：将预测结果转换为 AnnData 对象。
    # 运行目的：方便后续处理和提取结果。
    # 详细代码解析：
    # 1. `.to_adata()`
    #    - 将 `AnnotationResult` 对象转换为 `AnnData` 对象。
    #    - 预测结果（标签、概率等）存储在返回对象的 `.obs` 中。

    predictions_high_adata = predictions_high.to_adata(insert_conf_by="majority_voting")


...并将结果复制到我们原始的 AnnData 对象：

In [ ]:
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
if model_high is not None:
    # 功能说明：将粗略注释结果复制回原始 AnnData 对象。
    # 运行目的：将自动注释结果整合到主分析对象中，以便进行可视化和对比。
    # 详细代码解析：
    # 1. `adata.obs["celltypist_cell_label_coarse"]`
    #    - 存储多数投票后的预测标签 (`"majority_voting"`)。
    # 2. `adata.obs["celltypist_conf_score_coarse"]`
    #    - 存储预测的置信度分数 (`"conf_score"`)。

    adata.obs["celltypist_cell_label_coarse"] = predictions_high_adata.obs.loc[
        adata.obs.index, "majority_voting"
    ]
    adata.obs["celltypist_conf_score_coarse"] = predictions_high_adata.obs.loc[
        adata.obs.index, "conf_score"
    ]
    adata.obs["celltypist_predicted_label_coarse"] = predictions_high.predicted_labels["predicted_labels"].reindex(adata.obs_names)



现在对更精细的注释做同样的事情：

In [ ]:
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
if model_low is not None:
    # 功能说明：使用精细模型进行细胞类型预测。
    # 运行目的：自动注释数据中的细胞类型（精细级别）。
    # 详细代码解析：
    # 1. `celltypist.annotate(...)`
    #    - 使用 `model_low` 进行预测。

    predictions_low = celltypist.annotate(
        adata_celltypist, model=model_low, majority_voting=True
    )


In [ ]:
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
if model_low is not None:
    # 功能说明：将精细预测结果转换为 AnnData 对象。

    predictions_low_adata = predictions_low.to_adata(insert_conf_by="majority_voting")


In [ ]:
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
if model_low is not None:
    # 功能说明：将精细注释结果复制回原始 AnnData 对象。
    # 运行目的：整合精细注释结果。

    adata.obs["celltypist_cell_label_fine"] = predictions_low_adata.obs.loc[
        adata.obs.index, "majority_voting"
    ]
    adata.obs["celltypist_conf_score_fine"] = predictions_low_adata.obs.loc[
        adata.obs.index, "conf_score"
    ]
    adata.obs["celltypist_predicted_label_fine"] = predictions_low.predicted_labels["predicted_labels"].reindex(adata.obs_names)



现在绘图：

In [ ]:
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
if model_high is not None:
    # 功能说明：在 UMAP 上可视化粗略注释结果和置信度。
    # 运行目的：直观评估自动注释的效果和可靠性。
    # 详细代码解析：
    # 1. `sc.pl.umap(...)`
    #    - `color=["celltypist_cell_label_coarse", "celltypist_conf_score_coarse"]`:
    #      - 左图：展示粗略分类标签。
    #      - 右图：展示分类的置信度分数。颜色越深/亮表示置信度越高。
    #    - `wspace=1`: 增加子图间距。

    sc.pl.umap(
        adata,
        color=["celltypist_cell_label_coarse", "celltypist_conf_score_coarse"],
        frameon=False,
        sort_order=False,
        wspace=1,
        save="_11_303.pdf"
    )


In [ ]:
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
if model_low is not None:
    # 功能说明：在 UMAP 上可视化精细注释结果和置信度。
    # 运行目的：评估精细注释的效果。
    # 详细代码解析：
    # 1. `sc.pl.umap(...)`
    #    - `color=["celltypist_cell_label_fine", "celltypist_conf_score_fine"]`: 展示精细标签和置信度。

    sc.pl.umap(
        adata,
        color=["celltypist_cell_label_fine", "celltypist_conf_score_fine"],
        frameon=False,
        sort_order=False,
        wspace=1,
        save="_11_304.pdf"
    )


了解这些注释质量的一种方法是查看观察到的细胞类型相似性是否符合我们的预期：

In [ ]:
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
if model_low is not None:
    # 功能说明：绘制树状图（Dendrogram），展示预测细胞类型之间的转录组相似性。
    # 运行目的：检查自动注释的细胞类型在转录组水平上是否合理聚类。例如，不同类型的 T 细胞应该聚在一起。如果出现生物学上不合理的聚类（如 T 细胞与红细胞聚在一起），可能提示注释错误。
    # 详细代码解析：
    # 1. `sc.pl.dendrogram(adata, groupby="celltypist_cell_label_fine")`
    #    - `sc.pl.dendrogram`: 绘制树状图。
    #    - `groupby="celltypist_cell_label_fine"`: 计算给定分组（这里是精细注释标签）的平均表达谱，并基于相关性构建树状图。

    sc.pl.dendrogram(adata, groupby="celltypist_cell_label_fine", save="_11_306.pdf")


观察同类或相关细胞群是否具有相似的表达模式。树状图是转录组相似性的概括，不能单独证明发育谱系或断定标签正确。

现在让我们看看我们早期的手动注释：

In [ ]:
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
if model_high is not None:
    # 功能说明：在 UMAP 上对比手动注释和自动注释结果。
    # 运行目的：直观比较两种注释方法的一致性和差异。
    # 详细代码解析：
    # 1. `color=[...]`:
    #    - `manual_level2`: 手动二级注释。
    #    - `celltypist_cell_label_coarse`: 自动粗略注释。
    #    - `leiden_res_0_50`: 聚类结果。

    # 查看注释结果
    with rc_context({"figure.figsize": (10, 8)}):
        sc.pl.umap(
                adata,
                color=["manual_level2","celltypist_cell_label_coarse",fine_key ],
                legend_loc="on data",
                save="_11_309.pdf",
                )


比较手工注释与自动标签的一致性，并结合置信度和标记基因解释不一致区域。两者都不是独立的真实标签；一致并不等同于已验证准确。

下面选择精细模型多数投票标签置信度中位数最低的聚类，检查该簇内部的粗、细预测组成。

In [ ]:
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
confidence_column = "celltypist_conf_score_fine" if model_low is not None else "celltypist_conf_score_coarse"
focus_cluster = adata.obs.groupby(fine_key, observed=True)[confidence_column].median().idxmin()
print("本次检查的聚类：", focus_cluster)



In [ ]:
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
if model_low is not None:
    # 功能说明：查看特定簇（Cluster 4）在自动注释（精细）中的构成。
    # 运行目的：更细致地分析 Cluster 4 的成分。

    pd.crosstab(adata.obs[fine_key], adata.obs.celltypist_cell_label_fine).loc[
        focus_cluster, :
    ].sort_values(ascending=False)


自动注释为进一步审阅提供起点。对不一致或证据不足的标签保留不确定性，并利用已知标记基因、样本组成和模型适用范围继续检查。

## 保存本章结果

保存表格、参数摘要和可供后续章节读取的数据。

In [ ]:
# 变量/函数/参数解析：
# - label_choices：真实存在的手工或模型标签列；保留它们以便查分歧。
# - manual_vs_auto/confidence_by_cluster：交叉表和按簇分数，供 AI 作有证据的比较。
# - report_label_column：用户选定的报告主标签；12 只读取该列，不删除其他注释。
# 功能说明：并列保留手工与实际模型标签，复核分歧后确认报告主标签。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
ctx.table("celltypist_annotations", adata.obs)
confidence_columns = [c for c in adata.obs if c.startswith("celltypist_conf_score_")]
for level,manual in [("coarse","manual_level1"),("fine","manual_level2")]:
    column = "celltypist_cell_label_"+level
    if column in adata.obs:
        ctx.table("manual_vs_auto_"+level, pd.crosstab(adata.obs[manual],adata.obs[column]))
ctx.table("confidence_by_cluster", adata.obs.groupby(fine_key, observed=True)[confidence_columns].agg(["median","mean","min"]))
label_choices = {c: {"n_types": int(adata.obs[c].nunique())} for c in ["manual_level2","celltypist_cell_label_coarse","celltypist_cell_label_fine"] if c in adata.obs}
final_label = ctx.choose("report_labels", label_choices, "请阅读手工/模型分歧与预测分数，确认报告主标签；其他标签全部保留。", files=sorted(ctx.tables.glob("*.csv")))
adata.uns["report_label_column"] = final_label
ctx.finish(adata, {"models": model_specs, "report_label_column": final_label, "label_counts": adata.obs[final_label].value_counts().to_dict()})


## 结果阅读与思考

请打开本次结果目录中的 summary.json、tables 和 figures。将目的、方法、结果和解释写入本章报告源稿，再更新 Word。

思考题：自动注释与手工注释不一致时，应继续检查哪些证据？

运行与报告操作见课程根目录的 99_运行与AI协作指南.md。